# SageMaker Workflow :: Stakeholder Question 1 

"**Predict whether a 311 complaint will be resolved within 3 days.**"

Use this notebook to review and run a complete SageMaker **Linear Learner** workflow for this stakeholder question. This notebook is designed for the Day 28 in-class compare-and-contrast activity so that you can connect the SageMaker built-in algorithm workflow to the sklearn work you completed previously.

At a high level, this notebook will help you:
- Load a modeling dataset from S3,
- Preprocess the data into the format Linear Learner expects,
- Upload train and test files back to S3,
- Launch a SageMaker training job,
- Use Batch Transform to generate predictions, and
- Evaluate the model output in the notebook.

# Setup

This notebook uses SageMaker's built-in **Linear Learner** algorithm to train a baseline model for this stakeholder question. The overall workflow is different from sklearn: you will still prepare data in the notebook, but the actual training job runs on separately managed compute resources in SageMaker and reads its input data from S3.

In the cells below, you will:
- Import the libraries needed for SageMaker, pandas, and evaluation,
- Connect to your SageMaker session and execution role,
- Define your S3 bucket and folder paths, and
- Retrieve the correct regional container image for Linear Learner.

In [10]:
import io
import boto3
import sagemaker
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = session.boto_region_name
s3client = boto3.client('s3')

print(f"Region: {region}")
print(f"Role: {role}")

Region: us-east-1
Role: arn:aws:iam::533267250538:role/LabRole


## S3 configuration

Update the bucket and prefix below so they match the location of your exported modeling CSV. Keeping a clear folder structure in S3 matters because the training data, test data, model artifacts, and prediction outputs are all written to separate locations rather than staying inside notebook memory as they would in sklearn.

In [11]:
# ---------------------------------------------------------------
# Update these to match your S3 bucket and folder structure
# ---------------------------------------------------------------
S3_BUCKET = 'aws-nyc311-krame154-2026'
S3_PREFIX = 'modeling' # Modify as needed

LL_IMAGE = sagemaker.image_uris.retrieve('linear-learner', region)

INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


## Helper functions

Linear Learner expects a very specific CSV format: all features must be numeric, the label must be in the first column, and there should be no header row. These helper functions make it easier to upload properly formatted files to S3 and reload outputs such as batch prediction results.

In [12]:
def upload_csv_to_s3(df, bucket, key):
    """Upload a DataFrame as a headerless CSV to S3."""
    buf = io.BytesIO()
    df.to_csv(buf, index=False, header=False)
    buf.seek(0)
    s3client.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())
    print(f"Uploaded s3://{bucket}/{key} with {len(df):,} rows")
    return f"s3://{bucket}/{key}"

def download_csv_from_s3(bucket, key, **kwargs):
    """Download a CSV from S3 into a DataFrame."""
    obj = s3client.get_object(Bucket=bucket, Key=key)
    data = obj['Body'].read().decode('utf-8')
    return pd.read_csv(io.StringIO(data), **kwargs)

def make_ll_dataframe(X, y):
    """
    Combine label (y) and features (X) into the format Linear Learner expects:
    label first, all numeric, no header.
    """
    if hasattr(X, 'toarray'):
        X = X.toarray()
    label_col = pd.Series(y, name='label').reset_index(drop=True)
    feat_df = pd.DataFrame(X).reset_index(drop=True)
    return pd.concat([label_col, feat_df], axis=1)

## Load and preprocess the data

This stakeholder question is a **binary classification** problem. The target column is `resolved_quickly`, where 1 means the complaint was resolved within 3 days and 0 means it took longer. This notebook keeps the same modeling logic you already used with sklearn, but it reformats the data into the structure required by Linear Learner.

The main preprocessing step here is one-hot encoding the categorical predictors. SageMaker built-in algorithms do not accept string columns directly, so columns like agency and borough must be converted to numeric indicators before training.

In [13]:
# Update your path to your modeling data for this stakeholder question as needed
df = download_csv_from_s3(S3_BUCKET, "modeling/modeling_data.csv")
print(df.shape)
display(df.head())

# Features and target
cat_cols = ['agency', 'borough', 'problem_category']
num_cols = ['day_of_week', 'hour_of_day']
target = 'resolved_quickly'

X = df[cat_cols + num_cols]
y = df[target].astype(float)

# One-hot encode categoricals; pass numerics through
preprocessor = ColumnTransformer(
    transformers=[
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', 'passthrough', num_cols),
    ]
)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Training class balance:')
print(pd.Series(y_train).value_counts(normalize=True).round(3))

(10, 8)


,agency,borough,problem,incident_zip,day_of_week,hour_of_day,problem_category,resolved_quickly
0,HPD,BRONX,HEAT/HOT WATER,10460,4,8,housing,1
1,HPD,MANHATTAN,PAINT/PLASTER,10010,4,8,housing,1
2,NYPD,QUEENS,Noise - Vehicle,11372,4,8,other,1
3,DOT,BRONX,Traffic Signal Condition,10451,4,8,traffic,1
4,NYPD,QUEENS,Illegal Parking,11101,4,8,traffic,1


Train: (8, 14) Test: (2, 14)
Training class balance:
resolved_quickly
1.0    0.75
0.0    0.25
Name: proportion, dtype: float64


## Upload train and test files to S3

Even though you are working inside a notebook, SageMaker's managed training job cannot directly access variables sitting in notebook memory. That is why we convert the encoded arrays into headerless CSV files, place the label first, and upload the train and test data to S3 before training begins.

In [14]:
train_path = upload_csv_to_s3(
    make_ll_dataframe(X_train, y_train),
    S3_BUCKET,
    f"{S3_PREFIX}/ll-binary/train/train.csv",
)

test_path = upload_csv_to_s3(
    make_ll_dataframe(X_test, y_test),
    S3_BUCKET,
    f"{S3_PREFIX}/ll-binary/test/test.csv",
)

Uploaded s3://aws-nyc311-krame154-2026/modeling/ll-binary/train/train.csv with 8 rows
Uploaded s3://aws-nyc311-krame154-2026/modeling/ll-binary/test/test.csv with 2 rows


## Train the Linear Learner model

This cell defines the SageMaker estimator and submits the training job. Notice the key hyperparameter: `predictor_type='binary_classifier'` 

Use the code exactly as written below: it sets the predictor type to binary classification, tells SageMaker how many encoded features exist, and lets SageMaker handle feature normalization internally. This is the main point where the workflow diverges from sklearn's `model.fit(...)` pattern.

In [25]:
# Convert to numeric-only & label-first format

# Move target to first column
df_model = df.copy()

target = 'resolved_quickly'

cols = [target] + [c for c in df_model.columns if c != target]
df_model = df_model[cols]

# One-hot encode categorical columns
df_model = pd.get_dummies(df_model)

# Remove header + save locally
train_file = 'train_ll.csv'
df_model.to_csv(train_file, index=False, header=False)

In [26]:
import boto3

s3 = boto3.client('s3')

s3.upload_file(
    train_file,
    'aws-nyc311-krame154-2026',
    'modeling/train_ll.csv'
)

In [27]:
feature_dim = df_model.shape[1] - 1

In [28]:
print(df.head())
print(df_model.shape)

  agency    borough                   problem  incident_zip  day_of_week  \
0    HPD      BRONX            HEAT/HOT WATER         10460            4   
1    HPD  MANHATTAN             PAINT/PLASTER         10010            4   
2   NYPD     QUEENS           Noise - Vehicle         11372            4   
3    DOT      BRONX  Traffic Signal Condition         10451            4   
4   NYPD     QUEENS           Illegal Parking         11101            4   

   hour_of_day problem_category  resolved_quickly  
0            8          housing                 1  
1            8          housing                 1  
2            8            other                 1  
3            8          traffic                 1  
4            8          traffic                 1  
(10, 25)


In [29]:
df_model = df.copy()

target = 'resolved_quickly'

# Put label first
cols = [target] + [c for c in df_model.columns if c != target]
df_model = df_model[cols]

# One-hot encode strings into numbers
df_model = pd.get_dummies(df_model)

# Save no-header CSV
train_file = 'train_ll.csv'
df_model.to_csv(train_file, index=False, header=False)

feature_dim = df_model.shape[1] - 1

print(df_model.shape)
print(feature_dim)
df_model.head()

(10, 25)
24


,resolved_quickly,incident_zip,day_of_week,hour_of_day,agency_DCWP,agency_DOT,agency_DSNY,agency_HPD,agency_NYPD,borough_BRONX,...,problem_HEAT/HOT WATER,problem_Illegal Parking,problem_Noise - Vehicle,problem_PAINT/PLASTER,problem_Snow or Ice,problem_Traffic Signal Condition,problem_category_housing,problem_category_other,problem_category_sanitation,problem_category_traffic
0,1,10460,4,8,False,False,False,True,False,True,...,True,False,False,False,False,False,True,False,False,False
1,1,10010,4,8,False,False,False,True,False,False,...,False,False,False,True,False,False,True,False,False,False
2,1,11372,4,8,False,False,False,False,True,False,...,False,False,True,False,False,False,False,True,False,False
3,1,10451,4,8,False,True,False,False,False,True,...,False,False,False,False,False,True,False,False,False,True
4,1,11101,4,8,False,False,False,False,True,False,...,False,True,False,False,False,False,False,False,False,True


In [30]:
s3.upload_file(
    train_file,
    'aws-nyc311-krame154-2026',
    'modeling/train_ll.csv'
)

In [31]:
feature_dim=feature_dim

In [35]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput

train_path = "s3://aws-nyc311-krame154-2026/modeling/train_ll.csv"

ll_model = Estimator(
    image_uri=LL_IMAGE,
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=f"s3://{S3_BUCKET}/{S3_PREFIX}/ll-binary/output",
    sagemaker_session=session,
)

ll_model.set_hyperparameters(
    predictor_type='binary_classifier',
    feature_dim=feature_dim,
    mini_batch_size=2,
    epochs=10,
    normalize_data=True,
    normalize_label=False,
)

ll_model.fit(
    {'train': TrainingInput(train_path, content_type='text/csv')},
    logs=False
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: linear-learner-2026-04-29-05-04-43-897



2026-04-29 05:04:44 Starting - Starting the training job......
2026-04-29 05:05:22 Starting - Preparing the instances for training....
2026-04-29 05:05:45 Downloading - Downloading input data.......
2026-04-29 05:06:25 Downloading - Downloading the training image...................
2026-04-29 05:08:07 Training - Training image download completed. Training in progress...
2026-04-29 05:08:22 Uploading - Uploading generated training model..
2026-04-29 05:08:40 Completed - Training job completed


In [33]:
import boto3

s3 = boto3.client('s3')

for obj in s3.list_objects_v2(Bucket='aws-nyc311-krame154-2026')['Contents']:
    print(obj['Key'])

modeling/
modeling/ll-binary/output/linear-learner-2026-04-29-03-37-56-970/debug-output/training_job_end.ts
modeling/ll-binary/output/linear-learner-2026-04-29-03-37-56-970/profiler-output/framework/training_job_end.ts
modeling/ll-binary/output/linear-learner-2026-04-29-03-37-56-970/profiler-output/system/incremental/2026042903/1777433880.algo-1.json
modeling/ll-binary/output/linear-learner-2026-04-29-03-37-56-970/profiler-output/system/incremental/2026042903/1777433940.algo-1.json
modeling/ll-binary/output/linear-learner-2026-04-29-03-37-56-970/profiler-output/system/incremental/2026042903/1777434000.algo-1.json
modeling/ll-binary/output/linear-learner-2026-04-29-03-37-56-970/profiler-output/system/incremental/2026042903/1777434060.algo-1.json
modeling/ll-binary/output/linear-learner-2026-04-29-03-37-56-970/profiler-output/system/training_job_end.ts
modeling/ll-binary/output/linear-learner-2026-04-29-03-43-14-972/debug-output/training_job_end.ts
modeling/ll-binary/output/linear-learne

In [34]:
print(df_model.shape)

(10, 25)


## Run Batch Transform for predictions

Instead of deploying a real-time endpoint, this notebook uses **Batch Transform** to score the full test set. That is a good fit for your experimentation because SageMaker spins up temporary inference compute, writes predictions back to S3, and then shuts the job down when it is finished. This avoids leaving a persistent endpoint running, which would consume resources and create additional costs.

In [38]:
# -----------------------------
# Prepare test data correctly
# -----------------------------
test_df = pd.DataFrame(X_test)

# Align columns to match training (VERY IMPORTANT)
test_df = test_df.reindex(columns=df_model.columns[1:], fill_value=0)

# Save without header
test_file = "test_features.csv"
test_df.to_csv(test_file, index=False, header=False)

# Upload to S3
s3.upload_file(
    test_file,
    S3_BUCKET,
    f"{S3_PREFIX}/ll-binary/test/test-features.csv"
)

test_features_path = f"s3://{S3_BUCKET}/{S3_PREFIX}/ll-binary/test/test-features.csv"

# -----------------------------
# Run Batch Transform
# -----------------------------
transformer = ll_model.transformer(
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=f"s3://{S3_BUCKET}/{S3_PREFIX}/ll-binary/predictions",
    accept='text/csv',
    assemble_with='Line',
)

transformer.transform(
    test_features_path,
    content_type='text/csv',
    split_type='Line',
    wait=True,
)

print("Batch transform complete.")

INFO:sagemaker:Creating model with name: linear-learner-2026-04-29-05-23-06-252
INFO:sagemaker:Creating transform job with name: linear-learner-2026-04-29-05-23-06-926


...................................Docker entrypoint called with argument(s): serve
Running default environment configuration script
[04/29/2026 05:28:59 INFO 140328093017920] Memory profiler is not enabled by the environment variable ENABLE_PROFILER.
/opt/amazon/lib/python3.8/site-packages/mxnet/model.py:97: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if num_device is 1 and 'dist' not in kvstore:
/opt/amazon/lib/python3.8/site-packages/scipy/optimize/_shgo.py:495: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if cons['type'] is 'ineq':
/opt/amazon/lib/python3.8/site-packages/scipy/optimize/_shgo.py:743: SyntaxWarning: "is not" with a literal. Did you mean "!="?
  if len(self.X_min) is not 0:
[04/29/2026 05:29:03 WARNING 140328093017920] Loggers have already been setup.
[04/29/2026 05:29:03 INFO 140328093017920] loaded entry point class algorithm.serve.server_config:config_api
[04/29/2026 05:29:03 INFO 140328093017920] loading entry points
[04/29/2026 05:29:03 I

## Evaluate the predictions

The evaluation below mirrors the kind of performance review you should have already practiced with sklearn, including overall accuracy, F1 score, and a full classification report.

In [40]:
# Download predictions and evaluate
pred_key = f"{S3_PREFIX}/ll-binary/predictions/test-features.csv.out"
preds = download_csv_from_s3(S3_BUCKET, pred_key, header=None)
display(preds.head())

y_pred = preds.iloc[:, 0].astype(float)

print('Binary classification results')
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 (class 1): {f1_score(y_test, y_pred):.4f}")
print()
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=['Not quickly', 'Quickly'],
    zero_division=0
))

,0
0,1
1,1


Binary classification results
Accuracy: 1.0000
F1 (class 1): 1.0000

              precision    recall  f1-score   support

 Not quickly       0.00      0.00      0.00         0
     Quickly       1.00      1.00      1.00         2

    accuracy                           1.00         2
   macro avg       0.50      0.50      0.50         2
weighted avg       1.00      1.00      1.00         2



Acknowledgment:
I used an AI assistant (ChatGPT) to help debug errors related to SageMaker Linear Learner, including issues with data formatting, feature dimension mismatches, and batch transform setup. The assistant helped identify that SageMaker requires numeric-only, label-first CSV files without headers and that training and test data must have identical feature dimensions. All code was implemented, tested, and modified by me to fit the requirements of the assignment.

## Wrap-up

After you run the notebook, compare the results here to the sklearn workflow you used previously. As you review the outputs, pay attention to what changed conceptually: the model training happened as a SageMaker job, the data had to be staged in S3, and Batch Transform handled prediction without creating a persistent endpoint.

---
© Copyright 2026, The Department of Computational Mathematics, Science and Engineering at Michigan State University.